In [1]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24050256/versions/3 -O ../data/GeoClassing.zip
!mkdir ../data/GeoClassing
!unzip ../data/GeoClassing.zip -d ../data/GeoClassing/
!rm ../data/GeoClassing.zip

--2025-05-11 18:14:06--  https://data.ncl.ac.uk/ndownloader/articles/24050256/versions/3
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 52.48.143.184, 34.255.146.38, 54.216.15.6
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|52.48.143.184|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2651208517 (2.5G) [application/zip]
Saving to: ‘../data/GeoClassing.zip’

../data/GeoClassing 100%[===================>]   2.47G  45.8MB/s    in 89s     

2025-05-11 18:15:35 (28.3 MB/s) - ‘../data/GeoClassing.zip’ saved [2651208517/2651208517]

Archive:  ../data/GeoClassing.zip
 extracting: ../data/GeoClassing/BigEarthNetLicense.pdf  
 extracting: ../data/GeoClassing/test_x.npy  
 extracting: ../data/GeoClassing/test_y.npy  
 extracting: ../data/GeoClassing/train_x.npy  
 extracting: ../data/GeoClassing/train_y.npy  
 extracting: ../data/GeoClassing/valid_x.npy  
 extracting: ../data/GeoClassing/valid_y.npy  
 extracting: ../data/GeoClassing/README GeoClassing Dataset.txt  
 extra

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 60 * 60, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [4]:
hyperparams = {
    "num_epochs": 128,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.4},
    "pruning_thresholds": {"fc0": 0.1},
    "plateau_threshold": 0.5,
    "min_epochs_between_expansions": 24,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 1000,
    "weight_decay": 0,
}

"num_epochs: 128, batch_size: 256, metric: AbsGradientEdgeMetric, aggregation_mode: mean, choose_thresholds: {'fc0': 0.4}, choose_thresholds_del: {'fc0': 0.1}, threshold: 0.5, min_delta_epoch_replace: 24, window_size: 5, lr: 0.001, delete_after: 4, task_type: classification, fully_connected: False, max_to_replace: 1000, weight_decay: 0"

In [5]:
train_x = np.load("../data/GeoClassing/train_x.npy")
train_y = np.load("../data/GeoClassing/train_y.npy")
valid_x = np.load("../data/GeoClassing/valid_x.npy")
valid_y = np.load("../data/GeoClassing/valid_y.npy")
test_x = np.load('../data/GeoClassing/test_x.npy')
test_y = np.load('../data/GeoClassing/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 3 * 60 * 60)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 3 * 60 * 60)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 3 * 60 * 60)
test_tensor_y = torch.tensor(test_y).long()


train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)


train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [6]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [ ]:
import wandb

wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-GeoClassing",
    name=f"trash",
    config=hyperparams
)

In [8]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:02<00:00, 74.11it/s]


Epoch 1/128, Train Loss: 2.2419, Val Loss: 2.0589, Val Accuracy: 0.2156


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:01<00:00, 86.77it/s]


Epoch 2/128, Train Loss: 2.1263, Val Loss: 2.3472, Val Accuracy: 0.1793


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:01<00:00, 93.04it/s]


Epoch 3/128, Train Loss: 2.0952, Val Loss: 2.1803, Val Accuracy: 0.2156


... (246 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:06<00:00, 25.48it/s]


Epoch 127/128, Train Loss: 1.4982, Val Loss: 2.5198, Val Accuracy: 0.2821


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:06<00:00, 25.00it/s]


Epoch 128/128, Train Loss: 1.4194, Val Loss: 2.6282, Val Accuracy: 0.2590
--- 1475.5935604572296 seconds ---


In [9]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [10]:
accuracy, params

(0.2623700148554451, 1104920)

In [ ]:
torch.save(sparse_model, 'geoclassing.pt')

In [11]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 80.33, 92.49, 93.67, 72.76, 86.00, 94.21, 89.99, 94.61, 96.03, 95.66, 95.54, 63.56, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))
styled_table

Model,Accuracy (%),Parameters
Ours,26.24,"1,104,920.0"
ResNet-18,80.33,"11,689,512.0"
AlexNet,92.49,"61,100,840.0"
VGG16,93.67,"138,357,544.0"
ConvNext,72.76,"88,591,464.0"
MNASNet,86.00,"4,383,312.0"
DenseNet,94.21,"28,681,000.0"
ResNeXt,89.99,"25,028,904.0"
PC-DARTS,94.61,nan
DrNAS,96.03,nan
